In [ ]:
!pip install rasterio streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 59.0 MB/s eta 0:00:00


In [ ]:
# streamlit_app.py
import streamlit as st
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
import joblib

DEM Feature Extraction

In [ ]:
def compute_dem_features(dem, transform):
    elevation = dem.astype("float32")

    # slope
    x, y = np.gradient(elevation, transform.a, transform.e)
    slope = np.sqrt(x**2 + y**2)
    slope = np.degrees(np.arctan(slope))

    # hillshade (simple)
    azimuth_rad = 315 * np.pi / 180.0
    altitude_rad = 45 * np.pi / 180.0
    shaded = (np.cos(altitude_rad) * np.cos(np.radians(slope)) +
              np.sin(altitude_rad) * np.sin(np.radians(slope)) *
              (np.cos(azimuth_rad) * x + np.sin(azimuth_rad) * y))
    hillshade = shaded.clip(0, 1) * 255

    return slope, hillshade, elevation

In [ ]:
def resample_to_match(src_path, match_path):
    with rasterio.open(src_path) as src, rasterio.open(match_path) as match:
        data = src.read(
            1,
            out_shape=(match.height, match.width),
            resampling=rasterio.enums.Resampling.bilinear
        )
        return data, match.transform

Prediction Function

In [ ]:
def predict_raster(rf, x_tif_path, dem_tif_path):
    # Load Sentinel-2 bands
    with rasterio.open(x_tif_path) as src:
        X = src.read()
        profile = src.profile
    H, W = X.shape[1], X.shape[2]

    # Extract bands
    blue, green, red, nir, swir1, swir2 = X[:6]

    # Indices
    ndwi = (green - nir) / (green + nir + 1e-6)
    ndsi = (green - swir1) / (green + swir1 + 1e-6)
    mndwi = (green - swir2) / (green + swir2 + 1e-6)
    ndwi_blue = (blue - nir) / (blue + nir + 1e-6)

    # DEM features
    dem, transform = resample_to_match(dem_tif_path, x_tif_path)
    slope, hillshade, elevation = compute_dem_features(dem, transform)

    # Stack 13 features
    features = np.stack([
        blue, green, red, nir, swir1, swir2,
        ndwi, ndsi, mndwi, ndwi_blue,
        slope, hillshade, elevation
    ], axis=0)

    # Flatten
    X_flat = features.reshape(features.shape[0], -1).T
    mask = ~np.isnan(X_flat).any(axis=1)

    # Predict
    y_pred = np.zeros(X_flat.shape[0], dtype=np.uint8)
    y_pred[mask] = rf.predict(X_flat[mask])
    y_pred = y_pred.reshape(H, W)

    return y_pred

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%%writefile app.py
import streamlit as st
import matplotlib.pyplot as plt
import numpy as np
import rasterio
import joblib

st.title("🌊 Glacial Lake Mapping Tool")
st.write("Upload Sentinel-2 imagery (TIFF) and a DEM TIFF to map glacial lakes using a trained Random Forest model.")

# -------------------------------
# Prediction helper
# -------------------------------
def predict_raster(rf, x_tif_path, dem_tif_path):
    """Run Random Forest on Sentinel-2 + DEM to predict glacial lakes."""
    with rasterio.open(x_tif_path) as src:
        X = src.read()   # shape (bands, H, W)
        profile = src.profile
    H, W = X.shape[1], X.shape[2]
    blue, green, red, nir, swir1, swir2 = X[:6]

    # Indices
    ndwi = (green - nir) / (green + nir + 1e-6)
    ndsi = (green - swir1) / (green + swir1 + 1e-6)
    mndwi = (green - swir2) / (green + swir2 + 1e-6)
    ndwi_blue = (blue - nir) / (blue + nir + 1e-6)

    # DEM
    with rasterio.open(dem_tif_path) as dem_src:
        dem = dem_src.read(1, out_shape=(H, W))  # resample to match
    slope = np.gradient(dem)[0]
    hillshade = np.clip(200 - dem * 0.1, 0, 255)

    # Stack features (13)
    features = np.stack([
        blue, green, red, nir, swir1, swir2,
        ndwi, ndsi, mndwi, ndwi_blue,
        slope, hillshade, dem
    ], axis=0)

    X_flat = features.reshape(features.shape[0], -1).T
    mask = ~np.isnan(X_flat).any(axis=1)

    y_pred = np.zeros(X_flat.shape[0], dtype=np.uint8)
    y_pred[mask] = rf.predict(X_flat[mask])
    y_pred = y_pred.reshape(H, W)

    return y_pred

# -------------------------------
# File uploads
# -------------------------------
x_tif_file = st.file_uploader("Upload Sentinel-2 GeoTIFF", type=["tif", "tiff"])
dem_tif_file = st.file_uploader("Upload DEM GeoTIFF", type=["tif", "tiff"])

# Load pretrained RF model
rf_model_path = "/content/drive/MyDrive/GEE_Exports/rf_glacial_lake.pkl"
try:
    rf = joblib.load(rf_model_path)
    st.success("Random Forest model loaded successfully ✅")
except:
    st.error("⚠️ Random Forest model not found! Train and save as rf_glacial_lake.pkl")
    rf = None

# -------------------------------
# Run prediction
# -------------------------------
if x_tif_file and dem_tif_file and rf:
    with open("temp_x.tif", "wb") as f:
        f.write(x_tif_file.read())
    with open("temp_dem.tif", "wb") as f:
        f.write(dem_tif_file.read())

    st.info("Running prediction... ⏳")
    y_pred = predict_raster(rf, "temp_x.tif", "temp_dem.tif")

    st.write("### 🗺 Predicted Glacial Lake Map")
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(y_pred, cmap="Blues")
    ax.set_title("Glacial Lake Prediction")
    ax.axis("off")
    st.pyplot(fig)

Writing app.py


In [ ]:
import os
print(os.path.exists("/content/drive/MyDrive/GEE_Exports/rf_glacial_lake.pkl"))

True


In [ ]:
!apt-get install -q ngrok
!ngrok config add-authtoken 2xUkynJvKzihE6zGJFYpYN5lzNP_wU5TUpg2mdd2RJzVnwvt

Reading package lists...
Building dependency tree...
Reading state information...
E: Unable to locate package ngrok
Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
# ==============================
# Install dependencies
# ==============================
!pip install -q streamlit pyngrok

# Kill any previous Streamlit process (just in case)
!fuser -n tcp -k 8501 || echo "No previous process"

# ==============================
# Run Streamlit with ngrok
# ==============================
import subprocess
import time
from pyngrok import ngrok

# Launch Streamlit app in background
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])

# Wait for Streamlit to boot
time.sleep(5)

# Open ngrok tunnel
public_url = ngrok.connect(8501)
print("🌍 Your Streamlit app is live at:", public_url)

No previous process
🌍 Your Streamlit app is live at: NgrokTunnel: "https://5697c327eb85.ngrok-free.app" -> "http://localhost:8501"
